# 14 · 固定流程：工具查完，交给大模型判断

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → [14 固定流程 ✅] → 15 起：agent …
```

> **这一章的代码在合并请求 #16 里。** 调大模型的评测已经跑过（20 道题约 $0.03，200 道题约 $0.27），记在 MLflow 里。下面每一格都只读记录，**运行这本笔记不花钱**。

## 背景：数个数不懂的事

第 12 章按"解释了病人几个症状"重排，第 1 名答对率到了 30.5%。但数个数有三件事不懂：

1. **症状的分量不一样。** 一个特征性的化验结果（比如某种酶的活性降低）几乎能锁定一个病；"发育迟缓"上千个病都有。数个数时，它们都只算 1 个
2. **标志性表现缺了要扣分。** 第 12 章的组氨酸血症：50 个病人里 50 个血组氨酸升高，这位病人没有，这个病基本可以排除
3. **遗传方式、发病年龄、性别要对得上。** 比如 X 连锁的病，女孩得的可能性就小

这些要综合起来判断，正是大模型擅长的。这一章做计划里的对照组 **B2"固定流程"**：工具按固定顺序把证据查好，大模型只负责判断、写理由。

它和后面的 agent 只差一点：**查什么、查几次，是写死在代码里的**，模型不能自己决定。所以以后 agent 要和它比，才能说明"让模型自己决定"到底有没有用。

## 这一章做的 5 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 固定的三步流程 | 三个工具 + 一次模型调用 | 证据先查好，模型只判断 |
| 2 | 模型看到什么 | 一道真题的完整提示词 | 知道模型凭什么判断 |
| 3 | 先小考，再大考 | ci-20 → dev-200 | 先花 2 毛钱试通，再正式考 |
| 4 | 逐题比 | 配对比较 + 符号检验 | 两个百分比分不清时，看逐题输赢 |
| 5 | 看理由 | 模型改对、改错的题 | 每个答案都能查到"为什么" |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 固定的三步流程

### 在干什么
把三个工具按固定顺序串起来，最后问一次大模型。

### 背景
第 10–12 章做好了三个工具，第 12 章用"数个数"排了序。现在把排序后的前 10 名，连同它们的证据，交给大模型判断。

### 用了什么技术，为什么

```
病人的症状
  ① rank_diseases   8,000 多个病 → 前 30 名
  ② check_match     30 个逐条核对，按"解释了几个"重排（第 12 章）→ 取前 10 名
  ③ lookup_disease  查这 10 个病的基因、遗传方式、发病年龄
  ④ 大模型          读完 10 个病的证据，重新排序，每个写一句理由
```

几个决定：

| 决定 | 为什么 |
|---|---|
| 交给模型 10 个，不是 20、30 个 | 工具的前 10 名里有正确答案的占 54.5%，前 20 名 57.5%，前 30 名 59.5%。多给 10 个只多 3 个百分点，token 却翻倍 |
| 模型只能**重排**，不能加新的病 | 这一步只看"模型会不会判断证据"。模型漏排的候选，按工具原来的顺序接在后面 |
| 每个候选都要写一句理由 | 答案能解释、能核查（第 5 步） |
| 提示词写在代码里，有版本号 `pipeline-v1` | 和第 09 章一样：改提示词要审查、要重跑评测 |
| 关掉"思考模式"，温度 0 | 第 09 章比过：思考模式慢 20 倍、贵 25 倍，没更准 |

### 公司里什么情况怎么做
- **先做最简单的能工作的方案**：固定流程简单、便宜、每次都一样，出了问题好查。agent 更灵活也更难控制，要证明它比固定流程好才值得用
- **大模型只放在需要判断的那一步**：查找、核对这些有标准答案的事交给工具（第 12 章）

### 执行
从 MLflow 里读出这个系统正式跑分时记下的设置：

In [ ]:
from raredx.config import settings
from raredx.evals.harness import Harness

harness = Harness(settings.mlflow_tracking_uri, settings.mlflow_artifact_root)
def latest(name, split="dev-200"):
    return next(r for r in harness.runs(split) if r["name"] == name and not r["code"].endswith("-dirty"))

fp = latest("fixed-pipeline")
params = harness.client.get_run(fp["run_id"]).data.params
for key in ("model", "thinking", "prompt_version", "pool", "shown", "similarity", "hpo_release", "knowledge_mask"):
    print(f"{key:<16} {params[key]}")

### 结果怎么看
- `pool 30`、`shown 10`：工具先取 30 个，交给模型 10 个
- `prompt_version pipeline-v1`：用的是哪一版提示词
- `knowledge_mask case source paper`：和之前一样，每道题都遮住了它自己的论文（第 10 章）

### 面试怎么说
> "The fixed pipeline runs the tools in an order fixed in code — rank 30, check and reorder, look up the top 10 — and makes one model call to rank those ten on the evidence, with a reason for each. It is the control for the agent: same tools, no autonomy."

---

## 第 2 步 · 模型看到什么

### 在干什么
看一道真题发给模型的完整内容。

### 背景
模型判断得好不好，取决于它看到了什么。提示词就是它看到的全部。

### 用了什么技术，为什么
提示词分两部分：
- **系统提示**（每道题都一样）：身份是临床遗传学家；任务是给 10 个候选排序；判断原则（解释病人的症状，尤其是特异的；缺标志性表现、文献说没有的要扣分；遗传方式、发病年龄要对得上）；答案格式（JSON）
- **这道题**：性别、症状，然后 10 个候选，每个带着工具查到的证据

### 公司里什么情况怎么做
- **证据要精简**：每个候选只放判断需要的几行（第 11 章说过，回答要写给模型看）。这里每道题约 2,200 个 token
- **提示词里不放答案的任何线索**：没有论文号，没有正确的病名标记（有测试检查）

### 执行
用第 12 章那道 Aarskog-Scott 的题（它的资料不出自这道题的论文，没有泄漏），在本地生成这道题的提示词。只是生成文字，**不调用模型**：

In [ ]:
import mlflow
from raredx.baselines.fixed_pipeline import SHOWN, SYSTEM_PROMPT, user_prompt
from raredx.baselines.rank_then_check import RankThenCheck
from raredx.data import cases as case_store, sources
from raredx.evals.systems import question_of
from raredx.knowledge import hpo as hpo_knowledge, masking
from raredx.tools.check_match import CheckMatch
from raredx.tools.lookup_disease import LookupDisease
from raredx.tools.rank_diseases import RankDiseases

registry = sources.load_sources(ROOT / "evals/datasets.toml")
by_id = {c.id: c for c in case_store.read(sources.processed_dir(registry["phenopacket-store"], ROOT / "data"))}
hpo = hpo_knowledge.load(sources.raw_dir(registry["hpo"], ROOT / "data"))
tools = RankThenCheck(RankDiseases(hpo), CheckMatch(hpo))
lookup = LookupDisease(hpo)

case = by_id["PMID_38411716_Case_2"]
mlflow.tracing.disable()   # 只是生成文字，不需要记录
with masking.masked([case.pmid]):   # 和考试时一样遮住
    top = tools.ordered(question_of(case))[:SHOWN]
    shown = [(report, lookup(match.disease_id, max_features=1)) for match, report in top]
mlflow.tracing.enable()

prompt = user_prompt(question_of(case), shown)
print("\n".join(prompt.splitlines()[:16]))
print(f"…（后面还有 8 个候选）\n\n这道题：系统提示 {len(SYSTEM_PROMPT):,} 个字符 + 题目 {len(prompt):,} 个字符")

### 结果怎么看
- 最上面：性别（男）、病人的 13 个症状
- `[1]` Aarskog-Scott 和 `[2]` Stolerman 神经发育综合征，**都解释了 12/13 个症状**。只数个数，这两个分不出高下（第 12 章那个"咬合不正"只解释了 1 个，排到后面去了）
- 每个症状后面括号里，是这个病的病人里多少人有。差别在这里：Aarskog-Scott 的眼距宽 17/17、手掌宽 22/22、身材矮小 16/16；Stolerman 的身材矮小只有 1/67
- Aarskog-Scott 是 X 连锁隐性遗传，病人是男孩，对得上

模型最后把 Aarskog-Scott 排第 1，理由（原话）："explains 12 of 13 findings with high frequencies for key features like hypertelorism, broad palm, short stature… and X-linked recessive inheritance fits a male patient"；Stolerman 排第 2："weaker support for short stature"。**这正是数个数做不到的：看出现率、看遗传方式**

### 面试怎么说
> "Each candidate reaches the model as a few lines of evidence from the tools — what it explains and how often, what it doesn't, what its literature found absent, what the patient lacks — about 2,200 tokens per case, with nothing that points to the answer."

---

## 第 3 步 · 先小考，再大考

### 在干什么
先在 20 道题上试通，再在 200 道题上正式考。

### 背景
和第 09 章一样：调模型要花钱，先用小题集确认整条路是通的。

### 用了什么技术，为什么
跑过的两条命令（**已经跑过，不用再跑**）：

```
uv run raredx eval run --system fixed-pipeline --split ci-20     # 约 $0.03
uv run raredx eval run --system fixed-pipeline --split dev-200   # 约 $0.27
```

跑之前，先在本地把 200 道题的提示词全部生成一遍、量了长度，估出 dev-200 约 $0.24，确认了余额才跑。

### 公司里什么情况怎么做
- **花钱之前先估算**：提示词能在本地生成，长度能量，钱就能算个大概
- **小题集先跑**：路不通，只损失两毛钱

### 执行
从 MLflow 读出两次评测的成绩和花费：

In [ ]:
for split in ("ci-20", "dev-200"):
    record = latest("fixed-pipeline", split)
    m = harness.client.get_run(record["run_id"]).data.metrics
    print(f"{split:<8} 第 1 名 {m['disease_top1/mean']:.1%}  前 3 名 {m['disease_top3/mean']:.1%}  前 10 名 {m['disease_top10/mean']:.1%}"
          f"  | 答了 {m['n_answered']:.0f}/{m['n_cases']:.0f}  花费约 ${m['usage/cost_usd_estimate']:.2f}  每道题 {m['usage/seconds_per_case']:.1f} 秒")

### 结果怎么看
- **ci-20**：20 道题都答了，路是通的；20 道题的分数误差太大，不看高低
- **dev-200**：

| 系统 | 第 1 名 | 前 3 名 | 前 10 名 | 花费 |
|---|---|---|---|---|
| 先排名再核对（第 12 章） | 30.5% | 43.5% | 54.5% | 免费 |
| **固定流程（这一章）** | **35.5%** | **46.5%** | 54.5% | 约 $0.27 |

- **前 10 名没变**：模型只是把同样 10 个候选换了顺序，正确答案在不在这 10 个里，由工具决定
- **第 1 名多了 5 个百分点**。但两个误差范围有交叠（24.5–37.2% 和 29.2–42.3%）。这到底是真进步还是运气？下一步回答
- 200 道题都按格式答了，没有一道需要"补救"

### 面试怎么说
> "On dev-200 the fixed pipeline reaches 35.5% top-1 against 30.5% for the rule it builds on, at about $0.27 a run; top-10 is unchanged by construction, since the model only reorders the tools' ten."

---

## 第 4 步 · 逐题比：真进步还是运气？

### 在干什么
同样 200 道题，逐题看：哪些题只有固定流程答对，哪些只有第 12 章的规则答对。

### 背景
35.5% 和 30.5%，两个误差范围有交叠，只看两个百分比分不出来。但两个系统考的是**同一批题**，可以一道一道比，这比只看总分灵敏得多。

### 用了什么技术，为什么
**配对比较 + 符号检验**：
- 两个都对、两个都错的题，说明不了谁更好，不算
- 只看"意见不同"的题：一个对、一个错
- 如果两个系统其实一样好，意见不同的题里，应该各赢一半左右。输赢差得越悬殊，越不可能是碰巧。这个"碰巧的可能性"就是 p 值

这一步做成了一条命令 `raredx eval compare`，以后 agent 和固定流程比，也用它。

### 公司里什么情况怎么做
- **比较两个版本，一定在同一批题上逐题比**：A/B 对比的标准做法
- **报告输赢的题数，而不只是两个百分比**

### 执行

In [ ]:
rtc = latest("rank-then-check")
run(f"uv run raredx eval compare {rtc['run_id']} {fp['run_id']} --k 1")

### 结果怎么看
- **只有固定流程答对的：11 道；只有规则答对的：1 道**。两个都对 60 道，都错 128 道
- 意见不同的 12 道题里，11 比 1。如果两者一样好，出现这么悬殊的比分，可能性只有 0.6%（`p = 0.006`）。所以**这是真进步，不是运气**

### 面试怎么说
> "The rates' intervals overlap, so I paired them: on the same 200 cases the model-judged pipeline was right where the rule was wrong 11 times and the reverse once — a sign test gives p = 0.006."

---

## 第 5 步 · 看理由

### 在干什么
读几道题里模型写的理由：它改对的，和它改错的。

### 背景
数字之外，要读具体的题（第 09 章）。每个答案都带着一句理由，记在 MLflow 的追踪记录里。

### 执行

In [ ]:
from raredx.evals import scoring
from raredx.evals.harness import candidates_of

def answers(run_id):
    out = {}
    for t in harness.traces(run_id):
        root = t.data.spans[0]
        out[root.inputs["question"]["case_id"]] = candidates_of(root.outputs)
    return out

rule, model = answers(rtc["run_id"]), answers(fp["run_id"])
right = lambda answer, case: bool(answer) and scoring.normalize_omim(answer[0].disease_id) == case.disease_id
for title, keep in (("模型改对了", lambda c: right(model[c.id], c) and not right(rule[c.id], c)),
                    ("模型改错了", lambda c: right(rule[c.id], c) and not right(model[c.id], c))):
    print(f"—— {title} ——")
    for case in [by_id[i] for i in sorted(model) if keep(by_id[i])][:2]:
        print(f"正确答案：{case.disease_label}")
        print(f"  规则排第 1：{rule[case.id][0].disease_name}")
        print(f"  模型排第 1：{model[case.id][0].disease_name}")
        print(f"  模型的理由：{model[case.id][0].reason}\n")

### 结果怎么看
- **改对的**两道都是这种：病人有一组**特征性的检查异常**，数个数时每个只算 1 个，模型却知道它们几乎能锁定一个病。肌糖原贮积症 0 型那道：肌肉糖原降低、缺血运动后乳酸不升高（规则排第 2）；3-甲基巴豆酰辅酶 A 羧化酶缺乏症那道：模型说病人有这个病"标志性的三项化验异常"（规则排第 7）
- **改错的**那道（Leber 先天性黑蒙 6 型）：模型觉得发病年龄更像另一个视网膜病，把它排到了前面。理由写得清清楚楚，所以错在哪里也看得见

还要记住第 09 章说的：这些病例来自已发表的论文，模型训练时**可能读过**。这道题自己的论文虽然被遮住了，模型的记忆还在。所以 35.5% 里可能有一部分是"记得"，以后要用模型训练截止之后才发表的病例再考一次。

### 面试怎么说
> "The model's gains come where one finding is pathognomonic — a counted rule weighs it like any other — and every answer carries its reason, so its one loss is as legible as its wins."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 固定流程 | 三个工具 + 一次模型调用，顺序写死 | 最简单的能工作的方案；agent 的对照组 |
| 2 提示词 | 每个候选几行证据，约 2,200 token | 模型看到的就是判断的全部依据 |
| 3 小考、大考 | 先估算、再 ci-20、再 dev-200 | 花钱之前算清楚 |
| 4 逐题比 | `raredx eval compare`：配对 + 符号检验 | 两个百分比分不清时，看逐题输赢 |
| 5 理由 | 每个答案带一句理由，存在追踪记录里 | 对的错的都能解释 |

**一句话**：工具查好证据、大模型来判断，第 1 名答对率从 30.5% 提高到 35.5%；同一批题逐题比是 11 比 1，确实是进步；200 道题花了约 $0.27。

## 面试题

1. **What is your fixed-pipeline baseline, and why build it?**
   > "The same tools in an order fixed in code, plus one model call to judge the evidence. It isolates what the model's judgement adds, and it is the bar the agent's autonomy has to clear."

2. **How do you compare two systems whose scores are close?**
   > "On the same cases, paired: count the cases only one gets right and run a sign test. 35.5% against 30.5% looks inconclusive by interval; 11 to 1 on the same questions is p = 0.006."

3. **How do you control LLM cost before a run?**
   > "Build every prompt offline, measure it, multiply by the price; run 20 cases first. The estimate for this run was $0.24; it cost $0.27."

4. **What can't the fixed pipeline do?**
   > "Consider a disease the tools didn't propose. In nine dev cases the model alone named the right answer while the tools' top 30 missed it — an agent that can check its own hypotheses with the tools can reach those."

**下一章 15 · 第一个 agent：让大模型自己决定查什么**：同样三个工具，交给模型自己用。它可以先查自己想到的病、可以追问、可以查到觉得够了为止。它要在同一批题上，和这一章的固定流程逐题比。